# 6. Feature-family analysis

Five leave-one-family-out and five single-family models, retuned on the original grouped folds.

## Load fitted results

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "mea_ml" / "io.py").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display
from typing import Dict, List, Tuple, Any
from mea_ml.io import ROOT as PROJECT_ROOT, load_analysis
from mea_ml import training

OUT_DIR = PROJECT_ROOT / "results/06_family_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
bundle, artifact, lineage = load_analysis()
RANDOM_STATE = artifact["random_state"]

import warnings
from sklearn.exceptions import ConvergenceWarning


## Selected model and OOF predictions

In [2]:
X_map = {
    "mean+delta": bundle["X_mean_delta_df"].copy(),
    "delta-only": bundle["X_delta_only_df"].copy(),
    "mfr-only": bundle["X_mfr_only_df"].copy(),
}
feature_name_map = {
    "mean+delta": list(bundle["feature_names_mean_delta"]),
    "delta-only": list(bundle["feature_names_delta_only"]),
    "mfr-only": list(bundle["feature_names_mfr_only"]),
}
y = np.asarray(bundle["y"]).astype(int)
groups = np.asarray(bundle["groups"]).astype(str)
paired_meta = bundle["paired_meta"].copy()
feature_manifest = bundle["feature_manifest"].copy()
feature_catalog = bundle["feature_catalog"].copy()

selected_model = str(artifact["selected_model_name"])
selected_feature_set = str(artifact["selected_feature_set"])

X_primary = X_map[selected_feature_set].copy()
feature_names_primary = feature_name_map[selected_feature_set]
assert len(X_primary) == len(y) == len(groups) == len(paired_meta)

outer_fitted_models = artifact["outer_fitted_models"]
key = (selected_model, selected_feature_set)
if key not in outer_fitted_models:
    raise KeyError(f"Selected primary combination not found in outer_fitted_models: {key}")

selected_fold_models = outer_fitted_models[key]
N_REPEATS = int(artifact["n_repeats"])
OUTER_N_SPLITS = int(artifact["outer_n_splits"])
assert len(selected_fold_models) == N_REPEATS * OUTER_N_SPLITS
saved_repeat_oof = artifact["oof_predictions_by_repeat"].copy()
saved_repeat_oof = saved_repeat_oof[
    (saved_repeat_oof["model_name"] == selected_model) &
    (saved_repeat_oof["feature_set"] == selected_feature_set)
].copy()
assert not saved_repeat_oof.duplicated(["repeat_id", "sample_index"]).any()
print("N outer fold models:", len(selected_fold_models))
print("Primary X shape:", X_primary.shape)


N outer fold models: 15
Primary X shape: (90, 88)


## Feature families

In [3]:
raw_feature_to_family = dict(zip(feature_manifest["raw_feature"], feature_manifest["feature_family"]))

def engineered_feature_to_family(name):
    return raw_feature_to_family[name.split("::", 1)[1]]

primary_feature_family_map = pd.DataFrame({
    "feature_name": feature_names_primary,
    "feature_family": [engineered_feature_to_family(f) for f in feature_names_primary],
})
primary_feature_family_map.to_csv(OUT_DIR / "feature_family_map_primary.csv", index=False)


## Define family subsets

In [4]:
tuning_budget = dict(artifact["training_protocol"]["tuning_budget"])
selected_spec = next(s for s in training.make_model_specs() if s["model_name"] == selected_model)
subset_rows_by_groups = training.subset_rows
y_by_group = dict(zip(groups, y))
for group_id in np.unique(groups):
    assert len(np.unique(y[groups == group_id])) == 1
family_list = primary_feature_family_map["feature_family"].drop_duplicates().tolist()
assert "other" not in family_list, "Resolve unmapped features before family analysis."
family_to_features = {
    fam: primary_feature_family_map.loc[primary_feature_family_map["feature_family"] == fam, "feature_name"].tolist()
    for fam in family_list
}
family_tasks = []
for fam in family_list:
    family_tasks.append(("leave_one_family_out", fam, "drop_" + fam,
                         [f for f in X_primary.columns if f not in family_to_features[fam]]))
for fam in family_list:
    family_tasks.append(("family_only", fam, "only_" + fam, family_to_features[fam]))

family_protocol_key = joblib.hash({"lineage": lineage, "tasks": family_tasks})
outer_fold_specs = [{k: fm[k] for k in ["outer_fold_id", "repeat_id", "fold_in_repeat", "train_groups", "test_groups"]}
                    for fm in selected_fold_models]


## Nested evaluation of all subsets

In [5]:
def _evaluate_family_subset(analysis_type, family, subset_name, columns):
    if not columns:
        raise ValueError(f"Empty subset: {subset_name}")
    X_sub = X_primary[columns].copy()
    predictions, outer_rows, tuning_rows, split_rows = [], [], [], []
    for fm in outer_fold_specs:
        fid = int(fm["outer_fold_id"])
        rid = int(fm["repeat_id"])
        train_groups, test_groups = list(fm["train_groups"]), list(fm["test_groups"])
        X_train, train_mask = subset_rows_by_groups(X_sub, groups, train_groups)
        X_test, test_mask = subset_rows_by_groups(X_sub, groups, test_groups)
        assert set(train_groups).isdisjoint(test_groups)
        assert np.all(train_mask | test_mask) and not np.any(train_mask & test_mask)
        inner_seed = RANDOM_STATE + 100 * fid
        tuning_seed = RANDOM_STATE + 1000 * fid + selected_spec["simplicity_rank"]
        inner_folds = training.make_inner_folds(train_groups, y_by_group, seed=inner_seed)
        for inner_id, (tr, va) in enumerate(inner_folds, start=1):
            assert set(tr).isdisjoint(va)
            assert set(tr + va) == set(train_groups)
            assert set(tr + va).isdisjoint(test_groups)
            split_rows.append({"feature_subset_name": subset_name, "outer_fold_id": fid,
                "repeat_id": rid, "inner_fold_id": inner_id,
                "inner_train_groups": "|".join(tr), "inner_validation_groups": "|".join(va),
                "outer_test_groups": "|".join(test_groups)})
        best_params, best_score, trace = training.bayes_optimize_model(
            model_name=selected_model, estimator=selected_spec["estimator"], space=selected_spec["space"],
            X_df=X_train, y_arr=y[train_mask], groups_arr=groups[train_mask],
            inner_folds=inner_folds, seed=tuning_seed, **tuning_budget,
        )
        y_pred, y_proba, fitted_pipe = training.fit_predict_one_split(
            selected_model, selected_spec["estimator"], best_params, X_train, y[train_mask], X_test,
        )
        sample_indices = np.flatnonzero(test_mask)
        predictions.append(pd.DataFrame({
            "feature_subset_name": subset_name, "analysis_type": analysis_type, "feature_family": family,
            "repeat_id": rid, "outer_fold_id": fid, "fold_in_repeat": int(fm["fold_in_repeat"]),
            "sample_index": sample_indices, "group_id": groups[test_mask], "y_true": y[test_mask],
            "y_proba": y_proba, "y_pred": y_pred,
        }))
        scores = training.compute_metrics(y[test_mask], y_pred, y_proba)
        retained = (list(fitted_pipe.named_steps["corr"].selected_columns_)
                    if "corr" in fitted_pipe.named_steps else columns)
        outer_rows.append({"feature_subset_name": subset_name, "feature_family": family,
            "analysis_type": analysis_type, "outer_fold_id": fid, "repeat_id": rid,
            "fold_in_repeat": int(fm["fold_in_repeat"]), "n_input_features": len(columns),
            "n_retained_features": len(retained), "retained_features": "|".join(retained),
            "n_train": int(train_mask.sum()), "n_test": int(test_mask.sum()),
            "best_inner_cv_roc_auc": best_score, "best_params_json": json.dumps(best_params),
            "inner_seed": inner_seed, "tuning_seed": tuning_seed, **scores})
        trace = trace.copy()
        trace["feature_subset_name"], trace["outer_fold_id"], trace["repeat_id"] = subset_name, fid, rid
        tuning_rows.append(trace)
    pred_df = pd.concat(predictions, ignore_index=True)
    assert not pred_df.duplicated(["repeat_id", "sample_index"]).any()
    assert pred_df.groupby("sample_index").size().eq(N_REPEATS).all()
    assert pred_df.groupby("repeat_id").size().eq(len(y)).all()
    result = {"predictions": pred_df, "outer_metrics": pd.DataFrame(outer_rows),
              "tuning": pd.concat(tuning_rows, ignore_index=True), "inner_splits": pd.DataFrame(split_rows)}

    return result

def evaluate_family_subset(analysis_type, family, subset_name, columns):

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning,
                                module=r"sklearn\.gaussian_process\.kernels")
        return _evaluate_family_subset(analysis_type, family, subset_name, columns)

full_predictions = saved_repeat_oof[["repeat_id", "outer_fold_id", "fold_in_repeat", "sample_index",
                                    "group_id", "y_true", "y_proba", "y_pred"]].copy()
full_predictions["feature_subset_name"] = "full"
full_predictions["analysis_type"] = "full_baseline"
full_predictions["feature_family"] = "all"

family_evaluated = []
for task in family_tasks:
    print(f"Evaluating {task[2]}")
    family_evaluated.append(evaluate_family_subset(*task))
family_oof_by_repeat = pd.concat([full_predictions] + [r["predictions"] for r in family_evaluated], ignore_index=True)
family_outer_metrics = pd.concat([r["outer_metrics"] for r in family_evaluated], ignore_index=True)
family_tuning_detail = pd.concat([r["tuning"] for r in family_evaluated], ignore_index=True)
family_inner_splits = pd.concat([r["inner_splits"] for r in family_evaluated], ignore_index=True)

assert not family_oof_by_repeat.duplicated(["feature_subset_name", "repeat_id", "sample_index"]).any()

family_oof_mean = (family_oof_by_repeat.groupby(
    ["feature_subset_name", "analysis_type", "feature_family", "sample_index", "group_id", "y_true"], as_index=False)
    .agg(y_proba=("y_proba", "mean"), n_repeats=("repeat_id", "nunique")))
family_oof_mean["y_pred"] = (family_oof_mean["y_proba"] >= 0.5).astype(int)
assert family_oof_mean["n_repeats"].eq(N_REPEATS).all()
family_repeat_rows, family_summary_rows = [], []
for (name, rid), d in family_oof_by_repeat.groupby(["feature_subset_name", "repeat_id"]):
    metrics = training.compute_metrics(d["y_true"], d["y_pred"], d["y_proba"])
    family_repeat_rows.append({"feature_subset_name": name, "repeat_id": rid, **metrics})
family_repeat_metrics = pd.DataFrame(family_repeat_rows)
full_repeat_auc = family_repeat_metrics.query("feature_subset_name == 'full'").set_index("repeat_id")["roc_auc"]
family_repeat_metrics["delta_roc_auc_vs_full"] = family_repeat_metrics["roc_auc"] - family_repeat_metrics["repeat_id"].map(full_repeat_auc)
for (name, analysis_type, family), d in family_oof_mean.groupby(["feature_subset_name", "analysis_type", "feature_family"]):
    assert len(d) == len(y) and d["sample_index"].nunique() == len(y)
    metrics = training.compute_metrics(d["y_true"], d["y_pred"], d["y_proba"])
    rm = family_repeat_metrics[family_repeat_metrics["feature_subset_name"] == name]
    n_features = (len(X_primary.columns) if name == "full" else
                  next(len(t[3]) for t in family_tasks if t[2] == name))
    family_summary_rows.append({"feature_subset_name": name, "analysis_type": analysis_type,
        "feature_family": family, "n_input_features": n_features, "n_unique_wells": len(d),
        **{f"pooled_{k}": v for k, v in metrics.items()},
        "repeat_roc_auc_mean": rm["roc_auc"].mean(), "repeat_roc_auc_sd": rm["roc_auc"].std(ddof=1),
        "result_origin": "reused_from_03" if name == "full" else "nested_retuning"})
family_summary = pd.DataFrame(family_summary_rows)
full_auc = float(family_summary.loc[family_summary["feature_subset_name"] == "full", "pooled_roc_auc"].iloc[0])
family_summary["delta_roc_auc_vs_full"] = family_summary["pooled_roc_auc"] - full_auc
full_saved_mean = artifact["oof_predictions_main"]
full_saved_mean = full_saved_mean[(full_saved_mean["model_name"] == selected_model) &
                                  (full_saved_mean["feature_set"] == selected_feature_set)].sort_values("sample_index")
assert np.allclose(family_oof_mean.query("feature_subset_name == 'full'").sort_values("sample_index")["y_proba"],
                   full_saved_mean["y_proba"], atol=1e-12)
display(family_summary[["feature_subset_name", "n_input_features", "pooled_roc_auc", "delta_roc_auc_vs_full",
                        "repeat_roc_auc_mean", "repeat_roc_auc_sd"]])


Evaluating drop_burst
Evaluating drop_burst_timing
Evaluating drop_network_burst
Evaluating drop_spike_activity
Evaluating drop_synchrony_correlation
Evaluating only_burst
Evaluating only_burst_timing
Evaluating only_network_burst
Evaluating only_spike_activity
Evaluating only_synchrony_correlation


,feature_subset_name,n_input_features,pooled_roc_auc,delta_roc_auc_vs_full,repeat_roc_auc_mean,repeat_roc_auc_sd
0,drop_burst,68,0.896111,-0.013333,0.881944,0.033831
1,drop_burst_timing,62,0.859444,-0.050000,0.867222,0.032006
2,drop_network_burst,64,0.887778,-0.021667,0.877315,0.029641
3,drop_spike_activity,80,0.913333,0.003889,0.903241,0.021395
4,drop_synchrony_correlation,78,0.814444,-0.095000,0.781389,0.078535
5,full,88,0.909444,0.000000,0.895463,0.018659
6,only_burst,20,0.743889,-0.165556,0.745833,0.054111
7,only_burst_timing,26,0.795556,-0.113889,0.787500,0.020437
8,only_network_burst,24,0.720556,-0.188889,0.720648,0.010454
9,only_spike_activity,8,0.833889,-0.075556,0.807130,0.038499


## Save family comparisons

In [6]:
tables = {
    "family_performance_summary": family_summary,
    "family_ablation_leave_one_out_summary": family_summary[family_summary["analysis_type"].isin(["full_baseline", "leave_one_family_out"])],
    "family_only_summary": family_summary.query("analysis_type == 'family_only'"),
    "family_repeat_metrics": family_repeat_metrics, "family_oof_predictions_by_repeat": family_oof_by_repeat,
    "family_oof_predictions_mean": family_oof_mean, "family_outer_metrics": family_outer_metrics,
    "family_tuning_detail": family_tuning_detail, "family_inner_split_audit": family_inner_splits,
}
for stem, table in tables.items():
    table.to_csv(OUT_DIR / f"{stem}.csv", index=False)
analysis_summary = {
    "lineage": lineage, "training_random_state": RANDOM_STATE, "selected_model_name": selected_model, "selected_feature_set": selected_feature_set,
    "n_repeats": N_REPEATS, "outer_n_splits": OUTER_N_SPLITS, "n_unique_wells": len(y),
    "analysis_scope": "leave_one_family_out_and_family_only",
    "n_feature_families": len(family_list), "n_family_subsets": len(family_tasks),
    "tuning_budget": tuning_budget, "protocol_key": family_protocol_key,
    "selection": "each predefined family subset is retuned by inner-CV AUROC within outer training data",
    "aggregation": "average each well's 3 OOF probabilities; also export each repeat separately",
    "inference_scope": "exploratory after selected algorithm/representation; no full model-selection correction",
}
(OUT_DIR / "family_analysis_summary.json").write_text(json.dumps(analysis_summary, indent=2))
print("Saved family ablation and single-family comparisons:", OUT_DIR.relative_to(PROJECT_ROOT))


Saved family ablation and single-family comparisons: results/06_family_analysis
